In [5]:
# Train Model Notebook - Save this as notebooks/train_model.ipynb
# This generates a synthetic loan approval dataset and trains a classifier (XGBoost-based)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import os
import json
from xgboost import XGBClassifier

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic loan approval dataset
n_samples = 2000

data = {
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.randint(20000, 150000, n_samples),
    'loan_amount': np.random.randint(5000, 50000, n_samples),
    'credit_score': np.random.randint(300, 850, n_samples),
    'employment_years': np.random.randint(0, 40, n_samples),
    'debt_to_income': np.random.uniform(0, 0.8, n_samples),
    'gender': np.random.choice(['Male', 'Female'], n_samples),
    'ethnicity': np.random.choice(['White', 'Black', 'Hispanic', 'Asian'], n_samples)
}

df = pd.DataFrame(data)

# Create target variable (loan_approved) with intentional bias
def generate_loan_approval(row):
    score = 0
    
    # Credit score weight
    if row['credit_score'] > 700:
        score += 0.5
    
    # Income weight
    if row['income'] > 60000:
        score += 0.3
    
    # Debt to income
    if row['debt_to_income'] < 0.3:
        score += 0.2
    
    # Age weight
    if 25 <= row['age'] <= 55:
        score += 0.1
    
    # Intentional bias
    if row['gender'] == 'Male':
        score += 0.15  # bias towards males
    if row['ethnicity'] == 'White':
        score += 0.1  # bias towards whites
    
    score += np.random.normal(0, 0.15)
    return 1 if score > 0.6 else 0

df['loan_approved'] = df.apply(generate_loan_approval, axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nLoan approval rate: {df['loan_approved'].mean():.2%}")
print(f"\nApproval rate by gender:")
print(df.groupby('gender')['loan_approved'].mean())
print(f"\nApproval rate by ethnicity:")
print(df.groupby('ethnicity')['loan_approved'].mean())

# Save datasets
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/train.csv', index=False)
print("\n✅ Saved train.csv")

# Create test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
test_df.to_csv('../data/test.csv', index=False)
print("✅ Saved test.csv")

# Feature columns (excluding sensitive ones)
feature_cols = ['age', 'income', 'loan_amount', 'credit_score', 
                'employment_years', 'debt_to_income']

X_train = train_df[feature_cols]
y_train = train_df['loan_approved']
X_test = test_df[feature_cols]
y_test = test_df['loan_approved']

# ✅ Create and train XGBoost model pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

print("\n🔄 Training XGBoost model...")
pipeline.fit(X_train, y_train)

# Evaluate
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"✅ Model trained successfully using XGBoost!")
print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")

# Save model
os.makedirs('../models', exist_ok=True)
joblib.dump(pipeline, '../models/model.pkl')
print("\n✅ Model saved to models/model.pkl")

# Save feature info
feature_info = {
    'feature_names': feature_cols,
    'model_type': 'XGBoostClassifier',
    'trained_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('../models/model_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print("✅ Saved model_info.json")

# Save sample inputs for testing
sample_inputs = [
    {
        'age': 35,
        'income': 75000,
        'loan_amount': 25000,
        'credit_score': 720,
        'employment_years': 8,
        'debt_to_income': 0.25
    },
    {
        'age': 28,
        'income': 45000,
        'loan_amount': 15000,
        'credit_score': 650,
        'employment_years': 3,
        'debt_to_income': 0.45
    }
]

with open('../data/sample_inputs.json', 'w') as f:
    json.dump(sample_inputs, f, indent=2)

print("✅ Saved sample_inputs.json")
print("\n🎉 Phase 1 (Data + XGBoost Model) completed successfully!")


Dataset shape: (2000, 9)

Loan approval rate: 42.70%

Approval rate by gender:
gender
Female    0.333658
Male      0.525720
Name: loan_approved, dtype: float64

Approval rate by ethnicity:
ethnicity
Asian       0.399610
Black       0.425287
Hispanic    0.402464
White       0.483264
Name: loan_approved, dtype: float64

✅ Saved train.csv
✅ Saved test.csv

🔄 Training XGBoost model...


c:\itsMe\Projects\ai-bias-explainability\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:11:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Model trained successfully using XGBoost!
Train accuracy: 0.9994
Test accuracy: 0.8300

✅ Model saved to models/model.pkl
✅ Saved model_info.json
✅ Saved sample_inputs.json

🎉 Phase 1 (Data + XGBoost Model) completed successfully!
